# Export WIG Genus from HD5 to PNG Dataset

This notebook converts the WIG HDF5 archive into PNG files and a manifest for genus classification.

## 1. Configuration

These settings define the output folder, the minimum number of specimens per genus, and the train, validation, and test split.

In [ ]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


H5_PATH = Path("hd5_dataset/WIG_v1.2.1_600.h5")
OUTPUT_DIR = Path("wood_data")
MIN_SPECIMENS_PER_GENUS = 12
SPLITS = {"train": 0.70, "validation": 0.15, "test": 0.15}
RANDOM_SEED = 42

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_rows", 120)

## 2. Read HDF5 Metadata

The archive is organized as `family -> genus -> species -> specimen`. I create one row for each specimen stack with the fields needed for filtering and export.

In [2]:
specimen_rows = []

with h5py.File(H5_PATH, "r") as h5:
    for family, family_group in h5.items():
        for genus, genus_group in family_group.items():
            for species_label, species_group in genus_group.items():
                for specimen_id, image_stack in species_group.items():
                    specimen_rows.append({
                        "genus": genus,
                        "specimen_id": specimen_id,
                        "h5_path": f"{family}/{genus}/{species_label}/{specimen_id}",
                        "n_images": image_stack.shape[0],
                    })

specimens_all = (
    pd.DataFrame(specimen_rows)
    .sort_values(["genus", "h5_path"])
    .reset_index(drop=True)
)

display(specimens_all.head(10))
print(f"Found {len(specimens_all):,} specimen stacks with {specimens_all['n_images'].sum():,} images.")

,genus,specimen_id,h5_path,n_images
0,Acer,prep_09963,Sapindaceae/Acer/Acer_amoenum/prep_09963,11
1,Acer,prep_10073,Sapindaceae/Acer/Acer_amoenum/prep_10073,10
2,Acer,prep_09773,Sapindaceae/Acer/Acer_argutum/prep_09773,2
3,Acer,prep_09775,Sapindaceae/Acer/Acer_argutum/prep_09775,8
4,Acer,prep_03079,Sapindaceae/Acer/Acer_carpinifolium/prep_03079,1
5,Acer,prep_05998,Sapindaceae/Acer/Acer_carpinifolium/prep_05998,5
6,Acer,prep_06350,Sapindaceae/Acer/Acer_carpinifolium/prep_06350,7
7,Acer,prep_10018,Sapindaceae/Acer/Acer_carpinifolium/prep_10018,4
8,Acer,prep_10191,Sapindaceae/Acer/Acer_carpinifolium/prep_10191,8
9,Acer,prep_06215,Sapindaceae/Acer/Acer_crataegifolium/prep_06215,13


Found 540 specimen stacks with 7,051 images.


## 3. Filter Genera

I use genus as the target because species has too few independent specimens and family is too broad. Genera with at least 12 specimens are kept so that each class can be included in all three splits.

In [3]:
genus_summary = (
    specimens_all.groupby("genus")
    .agg(specimens=("specimen_id", "nunique"), images=("n_images", "sum"))
    .reset_index()
    .sort_values("genus")
)
genus_summary["status"] = np.where(
    genus_summary["specimens"] >= MIN_SPECIMENS_PER_GENUS,
    "retained",
    "excluded",
)

display(genus_summary)

retained_genera = genus_summary.loc[genus_summary["status"] == "retained", "genus"].tolist()
retained_specimens = specimens_all[specimens_all["genus"].isin(retained_genera)].copy()
retained_specimens = retained_specimens.sort_values(["genus", "h5_path"]).reset_index(drop=True)

print("Retained genera:", ", ".join(retained_genera))
print(f"Retained specimen count: {len(retained_specimens):,}")
print(f"Retained image count: {retained_specimens['n_images'].sum():,}")

,genus,specimens,images,status
0,Acer,34,295,retained
1,Actinodaphne,3,60,excluded
2,Aesculus,12,81,retained
3,Alnus,19,230,retained
4,Aphananthe,9,129,excluded
5,Beilschmiedia,1,20,excluded
6,Betula,12,136,retained
7,Carpinus,23,267,retained
8,Castanea,14,177,retained
9,Castanopsis,22,298,retained


Retained genera: Acer, Aesculus, Alnus, Betula, Carpinus, Castanea, Castanopsis, Cinnamomum, Fagus, Lindera, Lithocarpus, Litsea, Machilus, Magnolia, Quercus, Ulmus, Zelkova
Retained specimen count: 466
Retained image count: 5,992


## 4. Create Specimen-Level Splits

I split specimens within each genus using a fixed seed. All images from one specimen stay in the same split to avoid data leakage.

In [4]:
rng = np.random.default_rng(RANDOM_SEED)
split_rows = []

for genus, genus_specimens in retained_specimens.groupby("genus", sort=True):
    specimen_paths = genus_specimens["h5_path"].sort_values().to_numpy(copy=True)
    rng.shuffle(specimen_paths)

    n_specimens = len(specimen_paths)
    n_test = max(1, round(n_specimens * SPLITS["test"]))
    n_validation = max(1, round(n_specimens * SPLITS["validation"]))
    n_train = n_specimens - n_test - n_validation

    split_labels = (
        ["test"] * n_test
        + ["validation"] * n_validation
        + ["train"] * n_train
    )

    for h5_path, split in zip(specimen_paths, split_labels):
        split_rows.append({"h5_path": h5_path, "split": split})

split_assignments = pd.DataFrame(split_rows)
specimens = retained_specimens.merge(split_assignments, on="h5_path", how="left")
specimens = specimens.sort_values(["split", "genus", "h5_path"]).reset_index(drop=True)

specimen_split_table = pd.crosstab(specimens["genus"], specimens["split"]).reindex(
    columns=["train", "validation", "test"],
    fill_value=0,
)

display(specimen_split_table)

split,train,validation,test
genus,,,
Acer,24,5,5
Aesculus,8,2,2
Alnus,13,3,3
Betula,8,2,2
Carpinus,17,3,3
Castanea,10,2,2
Castanopsis,16,3,3
Cinnamomum,29,6,6
Fagus,17,3,3


## 5. Export PNG Files

I save each retained image as a grayscale PNG at its original 600 x 600 resolution. Resizing, normalization, and augmentation are handled in the modeling notebook.

In [5]:
for split in ["train", "validation", "test"]:
    for genus in retained_genera:
        (OUTPUT_DIR / split / genus).mkdir(parents=True, exist_ok=True)

manifest_rows = []

with h5py.File(H5_PATH, "r") as h5:
    for row in specimens.sort_values(["split", "genus", "h5_path"]).itertuples(index=False):
        output_folder = OUTPUT_DIR / row.split / row.genus
        image_stack = h5[row.h5_path]

        for image_index in range(row.n_images):
            filename = f"{row.specimen_id}_{image_index:03d}.png"
            output_path = output_folder / filename
            image_array = np.asarray(image_stack[image_index])
            Image.fromarray(image_array).save(output_path, format="PNG")

            manifest_rows.append({
                "filepath": output_path.relative_to(OUTPUT_DIR).as_posix(),
                "genus": row.genus,
                "specimen_id": row.specimen_id,
                "split": row.split,
                "image_index": image_index,
                "h5_path": row.h5_path,
            })

manifest = pd.DataFrame(manifest_rows).sort_values(
    ["split", "genus", "specimen_id", "image_index", "h5_path"]
).reset_index(drop=True)
manifest.to_csv(OUTPUT_DIR / "manifest.csv", index=False)

exported_files = sorted(OUTPUT_DIR.glob("*/*/*.png"))
export_counts = pd.Series({
    "Manifest rows": len(manifest),
    "Exported PNG files": len(exported_files),
}, name="count")

print(f"Saved the manifest to {OUTPUT_DIR / 'manifest.csv'}")
display(export_counts.to_frame())

Saved the manifest to wood_data\manifest.csv


,count
Manifest rows,5992
Exported PNG files,5992


## 7. Dataset Summary

In [6]:
split_order = ["train", "validation", "test"]
specimens_by_split = specimens["split"].value_counts().reindex(split_order).astype(int)
images_by_split = manifest["split"].value_counts().reindex(split_order).astype(int)
summary = pd.DataFrame({
    "specimens": specimens_by_split,
    "images": images_by_split,
})

print(f"Retained genera ({len(retained_genera)}): {', '.join(retained_genera)}")
print(f"Total retained specimens: {len(specimens):,}")
print(f"Total retained images: {len(manifest):,}")
print(f"Total exported PNG files: {len(exported_files):,}")
display(summary)

Retained genera (17): Acer, Aesculus, Alnus, Betula, Carpinus, Castanea, Castanopsis, Cinnamomum, Fagus, Lindera, Lithocarpus, Litsea, Machilus, Magnolia, Quercus, Ulmus, Zelkova
Total retained specimens: 466
Total retained images: 5,992
Total exported PNG files: 5,992


,specimens,images
split,,
train,328,4263
validation,69,836
test,69,893
